## Exercise 1 — Manual TF-IDF pipeline

Corpus: D1 *The Cat chased the Mouse.*, D2 *The dog Barked at the Cat.*, D3 *The mouse ran away from the Cat.*

Steps implemented below: lowercase, strip punctuation, tokenize (stop words **kept** for Ex1), vocabulary sorted, TF, normalized TF, IDF \(\log_{10}(N/(1+\mathrm{df}))\), TF–IDF = TF × IDF.

In [1]:
# Exercise 1 — TF-IDF (handout formulas: TF = f/sum(f); wtf = 1+log10(TF); IDF = log10(N/(1+df)); TF-IDF = TF*IDF)
import re
import numpy as np
import pandas as pd

docs_raw = [
    "The Cat chased the Mouse.",
    "The dog Barked at the Cat.",
    "The mouse ran away from the Cat.",
]

def tokenize(s):
    s = s.lower()
    s = re.sub(r"[^a-z\s]", " ", s)
    return [t for t in s.split() if t]

tokens_per_doc = [tokenize(d) for d in docs_raw]
print("Tokens (stop words kept, Exercise 1):")
for i, toks in enumerate(tokens_per_doc, 1):
    print(f"  D{i}: {toks}")

vocab = sorted({t for doc in tokens_per_doc for t in doc})
word2i = {w: i for i, w in enumerate(vocab)}
N_docs = len(tokens_per_doc)
V = len(vocab)

counts = np.zeros((N_docs, V))
for d, toks in enumerate(tokens_per_doc):
    for t in toks:
        counts[d, word2i[t]] += 1

TF = counts / counts.sum(axis=1, keepdims=True)

def wtf_matrix(tf):
    out = np.zeros_like(tf)
    m = tf > 0
    out[m] = 1 + np.log10(tf[m])
    return out

WTF = wtf_matrix(TF)
df = (counts > 0).sum(axis=0)
IDF = np.log10(N_docs / (1 + df))
TFIDF = TF * IDF

idx = [f"D{i+1}" for i in range(N_docs)]
print("\nVocabulary (sorted):", vocab)
print("\nTF:")
print(pd.DataFrame(TF, index=idx, columns=vocab).round(6))
print("\nNormalized TF (1 + log10 TF):")
print(pd.DataFrame(WTF, index=idx, columns=vocab).round(4))
print("\nIDF:")
print(pd.Series(IDF, index=vocab).round(4))
print("\nTF-IDF (TF × IDF):")
print(pd.DataFrame(TFIDF, index=idx, columns=vocab).round(4))

Tokens (stop words kept, Exercise 1):
  D1: ['the', 'cat', 'chased', 'the', 'mouse']
  D2: ['the', 'dog', 'barked', 'at', 'the', 'cat']
  D3: ['the', 'mouse', 'ran', 'away', 'from', 'the', 'cat']

Vocabulary (sorted): ['at', 'away', 'barked', 'cat', 'chased', 'dog', 'from', 'mouse', 'ran', 'the']

TF:
          at      away    barked       cat  chased       dog      from  \
D1  0.000000  0.000000  0.000000  0.200000     0.2  0.000000  0.000000   
D2  0.166667  0.000000  0.166667  0.166667     0.0  0.166667  0.000000   
D3  0.000000  0.142857  0.000000  0.142857     0.0  0.000000  0.142857   

       mouse       ran       the  
D1  0.200000  0.000000  0.400000  
D2  0.000000  0.000000  0.333333  
D3  0.142857  0.142857  0.285714  

Normalized TF (1 + log10 TF):
        at    away  barked     cat  chased     dog    from   mouse     ran  \
D1  0.0000  0.0000  0.0000  0.3010   0.301  0.0000  0.0000  0.3010  0.0000   
D2  0.2218  0.0000  0.2218  0.2218   0.000  0.2218  0.0000  0.0000  0.000

**Exercise 1 — Discussion**

1. **TF-IDF vs raw TF** — Raw TF favours repeated words (often function words). IDF down-weights terms that appear in many documents and up-weights rarer terms, so vectors highlight what is **distinctive** in each document relative to the corpus.

2. **Higher importance** — Large TF-IDF when TF is high **and** IDF is high (term is rare globally). Terms in all documents can get **negative** IDF with the handout’s smoothing formula \(\log_{10}(N/(1+\mathrm{df}))\).

3. **More documents** — \(N\) and each \(\mathrm{df}\) change, so IDFs and vector length (vocabulary) usually change; importance rankings shift.

In [4]:
# Exercise 2 — CBOW context–target pairs (window = 1, stop words removed per handout)
vocab_cbow = ["barked", "cat", "chased", "dog", "mouse", "ran"]
wi = {w: i for i, w in enumerate(vocab_cbow)}

def cbow_pairs(seq, window=1):
    rows = []
    for i, target in enumerate(seq):
        ctx = []
        if i - window >= 0:
            ctx.append(seq[i - window])
        if i + window < len(seq):
            ctx.append(seq[i + window])
        rows.append({
            "context_words": ctx,
            "context_indices": [wi[w] for w in ctx],
            "target": target,
            "target_index": wi[target],
        })
    return pd.DataFrame(rows)

D1 = ["cat", "chased", "mouse"]
D2 = ["dog", "barked", "cat"]
D3 = ["mouse", "ran", "cat"]

print("D1\n", cbow_pairs(D1), "\n")
print("D2\n", cbow_pairs(D2), "\n")
print("D3\n", cbow_pairs(D3), "\n")

oh = np.eye(len(vocab_cbow))
one_hot_df = pd.DataFrame(oh, index=vocab_cbow, columns=[f"h{i}" for i in range(len(vocab_cbow))])
print("One-hot rows:\n", one_hot_df.astype(int))

D1
   context_words context_indices  target  target_index
0      [chased]             [2]     cat             1
1  [cat, mouse]          [1, 4]  chased             2
2      [chased]             [2]   mouse             4 

D2
   context_words context_indices  target  target_index
0      [barked]             [0]     dog             3
1    [dog, cat]          [3, 1]  barked             0
2      [barked]             [0]     cat             1 

D3
   context_words context_indices target  target_index
0         [ran]             [5]  mouse             4
1  [mouse, cat]          [4, 1]    ran             5
2         [ran]             [5]    cat             1 

One-hot rows:
         h0  h1  h2  h3  h4  h5
barked   1   0   0   0   0   0
cat      0   1   0   0   0   0
chased   0   0   1   0   0   0
dog      0   0   0   1   0   0
mouse    0   0   0   0   1   0
ran      0   0   0   0   0   1


**Exercise 2 — Discussion**

1. **Same word, different contexts (e.g. “cat”)** — Each (context → target) pair is a separate training step. The embedding row for “cat” is updated from many different averaged context vectors, so it captures **aggregate** co-occurrence statistics, not one fixed neighbour list.

2. **Distributional hypothesis** — Words appearing in similar contexts tend to have similar meaning; CBOW learns vectors by predicting a word from its context, so words with overlapping contexts get **similar embeddings**.

### 3.2 CBOW architecture (summary)

- **Input layer:** one-hot size **V = 6** per context word (two context words → two one-hots).
- **Hidden layer:** embedding dim **N = 4**; vector = **average** of context rows from **W_input** (6×4). No activation.
- **Output layer:** **V = 6** logits = **v̂ @ W_hidden** with **W_hidden** shape **N×V (4×6)**; **softmax** → probabilities.
- **After training:** keep **W_input** rows as word embeddings; output weights are often discarded.

*(Handout typos: “chased” one-hot should be length 6, not 7; third printed logit 0.65 is **0.065** with the given weights.)*

In [6]:
# Tutorial 8 — CBOW forward pass (context ["cat", "mouse"] → predict "chased")
# W_input: 6×4, W_hidden: 4×6 (from module handout)

def softmax(x):
    x = np.asarray(x, dtype=float)
    e = np.exp(x - np.max(x))
    return e / e.sum()

W_input = np.array([
    [0.1, 0.2, -0.1, 0.0],
    [0.4, 0.5, -0.1, 0.2],
    [-0.2, 0.3, 0.2, -0.1],
    [0.3, -0.1, 0.1, 0.0],
    [0.5, -0.4, 0.1, 0.1],
    [-0.3, 0.0, -0.2, 0.3],
])
W_hidden = np.array([
    [0.2, -0.1, 0.3, 0.1, 0.4, 0.0],
    [0.1, 0.3, -0.2, -0.1, 0.2, 0.5],
    [-0.4, 0.5, 0.1, -0.2, -0.1, 0.3],
    [0.1, -0.2, -0.4, 0.3, -0.3, 0.1],
])

idx_cat, idx_mouse, idx_chased = 1, 4, 2
v_cat = W_input[idx_cat]
v_mouse = W_input[idx_mouse]
v_hat = (v_cat + v_mouse) / 2.0
# (N,) @ (N×V) → (V,) logits — matches handout dimensions
logits = v_hat @ W_hidden
probs = softmax(logits)
vocab6 = ["barked", "cat", "chased", "dog", "mouse", "ran"]

print("Average hidden vector v̂:", np.round(v_hat, 4))
print("Logits z:", np.round(logits, 4))
print("Softmax P(word|context):", np.round(probs, 4))
pred_i = int(np.argmax(logits))
print("Argmax (random init):", vocab6[pred_i], "| true target: chased — SGD + CE loss aligns prediction over training.")

Average hidden vector v̂: [0.45 0.05 0.   0.15]
Logits z: [ 0.11  -0.06   0.065  0.085  0.145  0.04 ]
Softmax P(word|context): [0.1741 0.1469 0.1665 0.1698 0.1803 0.1624]
Argmax (random init): mouse | true target: chased — SGD + CE loss aligns prediction over training.
